# Qwen 3.5 9B finetune on the Kaggle files: a reference notebook

This notebook runs the recipe of our Qwen 3.5 9B baseline from start to finish on the competition files:

1. It finetunes Qwen 3.5 9B with LoRA on train.csv.
2. It predicts with vLLM.
3. It scores a holdout locally with metric.py.
4. It writes a submission.csv for the leaderboard.

Use it as a reference point. Your own method has to classify with Apertus, but most of the pipeline carries over: the data loading, the image preparation, the loss on human labels only, the checks before training and the fast prediction.

The prompt here is a short one of its own, not our production prompt. So expect a score close to our baseline, not the same one.

**What you need**

- An H100 with 80 GB. On ours, training took 73 minutes with a peak of 38 GB, and prediction ran at about 29 images per second.
- About 100 GB of disk and 32 GB of RAM or more.
- A Kaggle account that has joined the competition and accepted its rules. The notebook logs in with a link.

The outputs are from that run. Run the cells from top to bottom and restart the kernel once after the install. The settings cell is the only one you need to edit. Never commit the data folder, the predictions or your Kaggle login: the data stays out of your repository.

## 1. Install

Run the three cells one by one, then restart the kernel. Unsloth patches transformers when it is imported, so upgrading it under a running kernel leaves a broken state.

The pins matter:

- Qwen 3.5 needs transformers 5.
- causal_conv1d 1.6.0 has prebuilt wheels for torch 2.8.0 only. Check the torch version that the first cell prints.
- pip reports that unsloth-zoo wants transformers 5.18 or older. Ignore it: the run works with 5.23.0.

In [1]:
import importlib.util
import subprocess
import sys


def pip_install(*arguments):
    """Install packages with the pip of this kernel."""
    command = [sys.executable, "-m", "pip", "install", "-q", *arguments]
    print(">", " ".join(command))
    subprocess.run(command, check=True)


# RunPod's PyTorch images ship torch, so it is only installed when missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch==2.8.0", "triton>=3.3.0", "numpy", "pillow", "torchvision", "xformers==0.0.32.post2")
pip_install("bitsandbytes")

import torch

print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

> /usr/local/bin/python -m pip install -q bitsandbytes
torch 2.8.0+cu128 | cuda 12.8
gpu NVIDIA H100 80GB HBM3


In [2]:
pip_install(
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo",
    "unsloth[base] @ git+https://github.com/unslothai/unsloth",
)
pip_install("--upgrade", "--no-deps", "tokenizers>=0.22.0,<=0.23.0", "trl==0.22.2", "unsloth", "unsloth_zoo")
pip_install("transformers==5.23.0")
pip_install("--no-deps", "--upgrade", "torchao>=0.16.0")
pip_install("huggingface_hub>=0.34.0", "hf_transfer", "kaggle==2.2.5", "pandas>=2.1", "py7zr")

> /usr/local/bin/python -m pip install -q unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo unsloth[base] @ git+https://github.com/unslothai/unsloth
> /usr/local/bin/python -m pip install -q --upgrade --no-deps tokenizers>=0.22.0,<=0.23.0 trl==0.22.2 unsloth unsloth_zoo
> /usr/local/bin/python -m pip install -q transformers==5.23.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.11.12 requires transformers!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.18.0,>=4.52.4, but you have transformers 5.23.0 which is incompatible.


> /usr/local/bin/python -m pip install -q --no-deps --upgrade torchao>=0.16.0
> /usr/local/bin/python -m pip install -q huggingface_hub>=0.34.0 hf_transfer kaggle==2.2.5 pandas>=2.1 py7zr


In [3]:
# The linear-attention kernels of Qwen 3.5.
pip_install("--no-build-isolation", "flash-linear-attention", "causal_conv1d==1.6.0")

# flash-linear-attention imports TileLang whenever it is installed, and TileLang needs libz3.so, which the RunPod
# image lacks. Without it the kernels fall back to Triton, and Qwen 3.5 needs nothing from TileLang.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "tilelang", "apache-tvm-ffi"], check=True)

> /usr/local/bin/python -m pip install -q --no-build-isolation flash-linear-attention causal_conv1d==1.6.0


CompletedProcess(args=['/usr/local/bin/python', '-m', 'pip', 'uninstall', '-y', '-q', 'tilelang', 'apache-tvm-ffi'], returncode=0)

### Restart the kernel now, then continue with the next cell

## 2. Settings

These values are the settings of our baseline. Two of them must match between training and prediction: `IMAGE_SIZE` and `JPEG_QUALITY`.

In [4]:
import importlib.util
import os
from pathlib import Path

WORK_DIR = Path("/workspace")  # any disk with about 100 GB free
os.environ["HF_HOME"] = str(WORK_DIR / "hf_cache")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

COMPETITION = "hackapertus-galaxus"
DATA_DIR = WORK_DIR / "data"
MODEL_NAME = "unsloth/Qwen3.5-9B"

# Images, the same in training and prediction. 512 means about 512 x 512 pixels: 256 vision tokens.
IMAGE_SIZE = 512
JPEG_QUALITY = 92

# Training data: a holdout of whole products for the local score, and repeats of rare classes.
HOLDOUT_IMAGES = 600
RARE_CLASS_TARGET = 200
MAX_REPEATS = 3

# LoRA and training. A batch of 4 x 4 fits on an H100 without gradient checkpointing.
LORA_RANK = 32
LORA_ALPHA = 32
NUM_TRAIN_EPOCHS = 3
LANGUAGE_LEARNING_RATE = 2e-4
VISION_LEARNING_RATE = 5e-5
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 2048
SEED = 3407

# Prediction. Add "final" on Saturday after 13:00, once the passphrase is out.
PREDICTED_PARTS = ("holdout", "dev")
VLLM_VERSION = "0.26.0"
VLLM_TRANSFORMERS_VERSION = "5.22.2"  # 5.23.0 breaks the memory profiling of vLLM for this model

ADAPTER_DIR = WORK_DIR / "qwen35_lora"
MERGED_DIR = WORK_DIR / "qwen35_merged"
OUTPUT_DIR = WORK_DIR / "outputs"
PREDICTION_DIR = WORK_DIR / "predictions"
VLLM_ENV_DIR = WORK_DIR / "vllm-env"
VLLM_LOG = WORK_DIR / "vllm-env.log"
VLLM_READY = WORK_DIR / "vllm-env.ready"
VLLM_FAILED = WORK_DIR / "vllm-env.failed"

assert IMAGE_SIZE % 32 == 0, "IMAGE_SIZE must be a multiple of 32"
assert importlib.util.find_spec("tilelang") is None, (
    "tilelang is installed: run the uninstall at the end of the install section and restart the kernel"
)

### vLLM in the background

vLLM and Unsloth want different torch versions, so vLLM gets its own virtual environment. The install takes several minutes and about 10 GB. It starts now and runs while the model trains. Running the cell again shows how far it is.

In [5]:
import subprocess
import sys

if VLLM_READY.exists():
    print(f"vLLM {VLLM_VERSION} is installed in {VLLM_ENV_DIR}")
elif VLLM_LOG.exists() and not VLLM_FAILED.exists():
    print(f"the vLLM install is still running, log: {VLLM_LOG}")
else:
    VLLM_FAILED.unlink(missing_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    uv = f"{sys.executable} -m uv"
    install_command = (
        f"{uv} venv {VLLM_ENV_DIR} --python 3.12 && "
        f"{uv} pip install --python {VLLM_ENV_DIR}/bin/python vllm=={VLLM_VERSION} "
        f"transformers=={VLLM_TRANSFORMERS_VERSION} --torch-backend=auto && "
        f"touch {VLLM_READY} || touch {VLLM_FAILED}"
    )
    with VLLM_LOG.open("w") as install_log:
        subprocess.Popen(
            ["bash", "-c", install_command], stdout=install_log, stderr=subprocess.STDOUT, start_new_session=True
        )
    print(f"installing vLLM {VLLM_VERSION} in the background, log: {VLLM_LOG}")

installing vLLM 0.26.0 in the background, log: /workspace/vllm-env.log


## 3. Download the data from Kaggle

The cell logs you in to Kaggle with a link:

1. Open the link it prints and log in.
2. Paste the code Kaggle shows you into the box below the cell.

The login stays on the pod for the next time. Ignore the help text about API tokens that appears first. Your account must have joined the competition and accepted its rules, otherwise the download fails with 403.

Then the cell downloads the competition files, about 5.8 GB, into `DATA_DIR`.

In [6]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
if not (Path.home() / ".kaggle" / "credentials.json").exists():
    api.auth_login_cli(no_launch_browser=True)
api.authenticate()
if not (DATA_DIR / "train.csv").exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    api.competition_download_files(COMPETITION, path=str(DATA_DIR), quiet=False, unzip=True)
print(sorted(path.name for path in DATA_DIR.iterdir()))

100%|██████████| 5.29G/5.29G [05:37<00:00, 16.8MB/s]  
['README.md', 'dev.csv', 'dev_images', 'final_encrypted.bin', 'labeling_guideline.md', 'labeling_guideline.pdf', 'metric.py', 'sample_submission.csv', 'train.csv', 'train_images']


## 4. Data, prompt and images

train.csv has one row per image, with the class of each dimension and two more columns per dimension:

- `<dimension>_source` is human or model. model marks a prelabel nobody checked.
- `<dimension>_relevant` is false where the dimension is not scored.

dev.csv, and later final.csv, have the same product columns without labels.

A holdout of whole products stays out of training and gets scored at the end. Holding out single images would make the score look better than it is, because other images of the same product would be in training.

**After a kernel restart:** run the settings cell and this section, then continue where you left off.

In [7]:
import random
import sys

import pandas as pd

sys.path.insert(0, str(DATA_DIR))
import metric  # the leaderboard metric, from the competition files

DIMENSIONS = metric.DIMENSIONS


def read_part(csv_path):
    """A csv of the competition as strings, with the full path of each image."""
    frame = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    frame["image_file"] = [str(csv_path.parent / image_path) for image_path in frame["image_path"]]
    return frame


labeled_rows = read_part(DATA_DIR / "train.csv")
dev_rows = read_part(DATA_DIR / "dev.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype=str, keep_default_na=False)

images_per_product = labeled_rows["product_id"].value_counts()
shuffled_products = sorted(images_per_product.index)
random.Random(SEED).shuffle(shuffled_products)
holdout_products, held_out_images = set(), 0
for product_id in shuffled_products:
    if held_out_images >= HOLDOUT_IMAGES:
        break
    holdout_products.add(product_id)
    held_out_images += images_per_product[product_id]

is_holdout = labeled_rows["product_id"].isin(holdout_products)
train_rows = labeled_rows[~is_holdout].reset_index(drop=True)
holdout_rows = labeled_rows[is_holdout].reset_index(drop=True)
print(
    f"train {len(train_rows)} images | holdout {len(holdout_rows)} images of {len(holdout_products)} products | "
    f"dev {len(dev_rows)} images"
)

train 11276 images | holdout 600 images of 378 products | dev 5261 images


### Prompt and answer

The prompt is short on purpose. After finetuning, the model knows the classes from the labels, and the prompt only frames the task.

The answer is a JSON object with the six dimensions in a fixed order. In prediction, vLLM is held to that schema, so every answer parses.

In [8]:
import json

CLASSES = {dimension: sorted(metric.VALID_CLASSES[dimension]) for dimension in DIMENSIONS}
DIMENSION_MEANINGS = {
    "subject": "what the image shows",
    "background": "the physical backdrop",
    "perspective": "from where the product is seen, judged by where its front points",
    "individuals": "a real person or animal in the photo",
    "text_classification": "text or elements added digitally on top of the photo, or a nutrition label",
    "text_language": "the dominant language of the visible text",
}
SYSTEM_PROMPT = (
    "You classify one product image of an online shop along six dimensions. "
    "Answer with a JSON object that holds one class per dimension, in this order:\n"
    + "\n".join(
        f"- {dimension} ({DIMENSION_MEANINGS[dimension]}): {', '.join(CLASSES[dimension])}" for dimension in DIMENSIONS
    )
)
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {dimension: {"type": "string", "enum": CLASSES[dimension]} for dimension in DIMENSIONS},
    "required": list(DIMENSIONS),
    "additionalProperties": False,
}


def user_prompt(row):
    """The product context of an image."""
    return (
        f"Product: {row['product_title']}\nBrand: {row['brand']}\nProduct type: {row['product_type']}\n"
        f"Category: {row['category']} / {row['subcategory']}"
    )


def serialize_answer(target):
    """The answer JSON, with the character span of each class value inside its quotes."""
    text, spans = "{", {}
    for index, (dimension, value) in enumerate(target.items()):
        text += ("" if index == 0 else ", ") + json.dumps(dimension) + ": "
        encoded = json.dumps(value)
        spans[dimension] = (len(text) + 1, len(text) + len(encoded) - 1)
        text += encoded
    return text + "}", spans


def parse_answer(answer):
    """The class per dimension of a model answer, or None when it is not a valid answer."""
    try:
        parsed = json.loads(answer[answer.find("{") : answer.rfind("}") + 1])
    except ValueError:
        return None
    if not isinstance(parsed, dict) or any(parsed.get(dimension) not in CLASSES[dimension] for dimension in DIMENSIONS):
        return None
    return {dimension: parsed[dimension] for dimension in DIMENSIONS}


def read_predictions(part):
    """The parsed answer per image of a predicted part, empty when the part was not predicted yet."""
    path = PREDICTION_DIR / f"predictions_{part}.jsonl"
    if not path.exists():
        return {}
    records = map(json.loads, path.read_text().splitlines())
    return {record["image_id"]: parse_answer(record["answer"]) for record in records}


print(SYSTEM_PROMPT)

You classify one product image of an online shop along six dimensions. Answer with a JSON object that holds one class per dimension, in this order:
- subject (what the image shows): accessories, delivery_scope, detail_closeup, energy_label, full_product, infographic, logo, multipack_trade_unit, multiview, package_and_product, packaging, placeholder, product_grouping
- background (the physical backdrop): color, scene_lifestyle, white
- perspective (from where the product is seen, judged by where its front points): back_angle_left, back_angle_right, back_head_on, bottom_up, front_angle_left, front_angle_right, front_head_on, left_side, right_side, top_down
- individuals (a real person or animal in the photo): individual_present, none
- text_classification (text or elements added digitally on top of the photo, or a nutrition label): certification_seal, none, nutritional_info, other, promotional_text, technical_specs
- text_language (the dominant language of the visible text): Dutch, Engli

### Images

Every image is prepared the same way for training and prediction:

1. Transparency goes onto white, as the shop shows it. A plain `convert("RGB")` turns it black.
2. The image is scaled to about `IMAGE_SIZE`² pixels. It keeps its aspect ratio, and both sides are a multiple of 32.
3. It is saved as JPEG.

So training sees exactly the pixels that vLLM gets later. Qwen 3.5 makes one vision token per 32 x 32 pixels.

In [9]:
import io

from PIL import Image as PillowImage
from PIL import ImageOps

PATCH_SIZE = 32
MAX_SIDE_FACTOR = 8


def opaque_rgb_on_white(image):
    """RGB, with any transparency composited onto white."""
    if image.mode not in ("RGBA", "LA", "PA") and "transparency" not in image.info:
        return image.convert("RGB")
    rgba = image.convert("RGBA")
    canvas = PillowImage.new("RGBA", rgba.size, (255, 255, 255, 255))
    canvas.alpha_composite(rgba)
    return canvas.convert("RGB")


def native_aspect_image(image, image_size):
    """The image scaled to about image_size squared pixels, aspect ratio kept, each side a multiple of the patch size."""
    scale = image_size / (image.width * image.height) ** 0.5
    scale *= min(1.0, MAX_SIDE_FACTOR * image_size / (max(image.width, image.height) * scale))
    width = max(PATCH_SIZE, round(image.width * scale / PATCH_SIZE) * PATCH_SIZE)
    height = max(PATCH_SIZE, round(image.height * scale / PATCH_SIZE) * PATCH_SIZE)
    return image.resize((width, height), PillowImage.LANCZOS)


def prepared_jpeg(image_file, mirrored=False):
    """The image as the model gets it, as JPEG bytes, mirrored first if asked."""
    with PillowImage.open(image_file) as image:
        rgb = opaque_rgb_on_white(image)
    if mirrored:
        rgb = ImageOps.mirror(rgb)
    buffer = io.BytesIO()
    native_aspect_image(rgb, IMAGE_SIZE).save(buffer, format="JPEG", quality=JPEG_QUALITY)
    return buffer.getvalue()


def prepare_image(image_file, mirrored=False):
    """The prepared JPEG decoded again: exactly the pixels vLLM gets."""
    return PillowImage.open(io.BytesIO(prepared_jpeg(image_file, mirrored))).convert("RGB")


def build_messages(row, image, answer):
    """The training conversation of an image: system prompt, image with product context, answer."""
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": user_prompt(row)}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]

## 5. Model and training examples

The model is trained with bf16 LoRA on both the language model and the vision tower.

In [10]:
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=False,
    load_in_16bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
    use_gradient_checkpointing=False,
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    target_modules="all-linear",
    use_gradient_checkpointing=False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.12: Fast Qwen3_5 patching. Transformers: 5.23.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.181 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: compiling each transformer block as one region (Qwen3_5DecoderLayer).


### What the loss sees

About a fifth of the label cells in train.csv are model prelabels, mostly where the dimension is not relevant. The model still writes the full JSON, but the class values of prelabels stay out of the loss. So it learns only from human labels, and it doesn't copy the biases of the old model.

Two more things lift macro F1:

- **Mirrored copies.** An image with a left or right perspective gets a mirrored copy, with left and right swapped in its label. Only images without text get one, because mirrored text is garbled.
- **Repeats.** Rows of rare classes are repeated up to `MAX_REPEATS` times per epoch.

In [11]:
import math
from collections import Counter

MIRRORED_PERSPECTIVES = {
    "front_angle_left": "front_angle_right",
    "front_angle_right": "front_angle_left",
    "back_angle_left": "back_angle_right",
    "back_angle_right": "back_angle_left",
    "left_side": "right_side",
    "right_side": "left_side",
}


def is_mirrorable(target, human_dimensions):
    """A human left or right perspective on an image without text, which a mirror would garble."""
    return (
        "perspective" in human_dimensions
        and target["perspective"] in MIRRORED_PERSPECTIVES
        and target["text_classification"] == "none"
        and target["text_language"] == "none"
    )


def mirrored_target(target):
    """The answer for the mirrored image: left and right swapped in the perspective."""
    return {**target, "perspective": MIRRORED_PERSPECTIVES[target["perspective"]]}


train_records = train_rows.to_dict("records")
targets = [{dimension: row[dimension] for dimension in DIMENSIONS} for row in train_records]
human = [[dimension for dimension in DIMENSIONS if row[f"{dimension}_source"] == "human"] for row in train_records]
class_counts = Counter(
    (dimension, target[dimension]) for target, dimensions in zip(targets, human) for dimension in dimensions
)


def repeat_count(target, dimensions):
    """How often a row is shown per epoch: enough to lift its rarest class towards RARE_CLASS_TARGET."""
    rarest = min((class_counts[(dimension, target[dimension])] for dimension in dimensions), default=RARE_CLASS_TARGET)
    return min(MAX_REPEATS, max(1, math.ceil(RARE_CLASS_TARGET / rarest)))


repeats = [repeat_count(target, dimensions) for target, dimensions in zip(targets, human)]
mirrored = [is_mirrorable(target, dimensions) for target, dimensions in zip(targets, human)]

for dimension in DIMENSIONS:
    human_count = sum(dimension in dimensions for dimensions in human)
    prelabel_count = len(targets) - human_count
    print(f"{dimension:20} {human_count:6} human labels in the loss | {prelabel_count:6} prelabels left out")
rows_per_epoch = sum(repeats) + sum(mirrored)
total_steps = math.ceil(rows_per_epoch / (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_TRAIN_EPOCHS
print(
    f"\nrows per epoch: {len(targets)} images + {sum(repeats) - len(targets)} repeats + {sum(mirrored)} mirrored "
    f"= {rows_per_epoch}, {total_steps} steps in total"
)

subject               10975 human labels in the loss |    301 prelabels left out
background             9409 human labels in the loss |   1867 prelabels left out
perspective            7704 human labels in the loss |   3572 prelabels left out
individuals            9340 human labels in the loss |   1936 prelabels left out
text_classification   10171 human labels in the loss |   1105 prelabels left out
text_language          5290 human labels in the loss |   5986 prelabels left out

rows per epoch: 11276 images + 1196 repeats + 1834 mirrored = 14306, 2685 steps in total


### Decode the images

Every image is decoded once and kept in RAM. The repeats share it, and a mirrored copy is one more image. This takes a few minutes.

In [12]:
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm


def build_example(job):
    """One training example: the conversation, plus the answer and the dimensions the loss sees."""
    index, flipped = job
    target = mirrored_target(targets[index]) if flipped else targets[index]
    answer = serialize_answer(target)[0]
    image = prepare_image(train_records[index]["image_file"], flipped)
    return {
        "messages": build_messages(train_records[index], image, answer),
        "answer": answer,
        "trained_dimensions": human[index],
    }


jobs = [(index, False) for index in range(len(train_records))]
jobs += [(index, True) for index, is_mirrored in enumerate(mirrored) if is_mirrored]
with ThreadPoolExecutor(max_workers=16) as pool:
    built_examples = list(tqdm(pool.map(build_example, jobs), total=len(jobs), desc="images"))

train_examples = []
for (index, flipped), example in zip(jobs, built_examples):
    train_examples.extend([example] if flipped else [example] * repeats[index])
del built_examples

sizes = Counter(example["messages"][1]["content"][0]["image"].size for example in train_examples)
largest_token_count = max((width // PATCH_SIZE) * (height // PATCH_SIZE) for width, height in sizes)
print(f"{len(train_examples)} rows per epoch, {len(sizes)} image sizes, at most {largest_token_count} vision tokens")

14306 rows per epoch, 133 image sizes, at most 306 vision tokens


## 6. Check, then train

### Collator

Unsloth's collator keeps only the answer in the loss. The subclass also masks the class values of prelabels; the keys, quotes and commas stay in, so the model still learns to write the whole JSON.

Two settings matter:

- `resize="max"` keeps our image sizes. The default shrinks the images.
- `pad_to_multiple_of=64` keeps the number of sequence lengths small. The linear-attention layers of Qwen 3.5 are compiled per sequence length, so without it every new length recompiles and a step takes minutes.

In [13]:
from unsloth.trainer import UnslothVisionDataCollator

text_tokenizer = tokenizer.tokenizer


def find_last_subsequence(sequence, subsequence):
    """Start index of the last occurrence of subsequence in sequence."""
    for start in range(len(sequence) - len(subsequence), -1, -1):
        if sequence[start : start + len(subsequence)] == subsequence:
            return start
    raise ValueError("the answer's tokens are not in the sequence, so the class values cannot be located")


def value_token_positions(input_ids, answer):
    """Sequence positions of each dimension's class value tokens, found through the answer at the end."""
    text, spans = serialize_answer(json.loads(answer))
    encoding = text_tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    start = find_last_subsequence(input_ids, encoding["input_ids"])
    return {
        dimension: [
            start + index
            for index, (token_start, token_end) in enumerate(encoding["offset_mapping"])
            if token_start < value_end and token_end > value_start
        ]
        for dimension, (value_start, value_end) in spans.items()
    }


class HumanLabelVisionDataCollator(UnslothVisionDataCollator):
    """Unsloth's vision collator, with the class values of prelabels left out of the loss."""

    def collate(self, examples):
        """The batch, plus per example the sequence positions of each dimension's class value."""
        batch = super().__call__([{"messages": example["messages"]} for example in examples])
        positions = []
        for row, example in enumerate(examples):
            example_positions = value_token_positions(batch["input_ids"][row].tolist(), example["answer"])
            for dimension, dimension_positions in example_positions.items():
                if dimension not in example["trained_dimensions"]:
                    batch["labels"][row, dimension_positions] = -100
            positions.append(example_positions)
        return batch, positions

    def __call__(self, examples):
        return self.collate(examples)[0]


collator = HumanLabelVisionDataCollator(
    model,
    tokenizer,
    resize="max",
    pad_to_multiple_of=64,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

### Check before you spend GPU hours

- **The think block.** Qwen 3.5 is a reasoning model. The prediction prompt must end with a closed think block, as in training. With an open one, the model writes pages of reasoning instead of the JSON.
- **The mask.** Human class values are in the loss, prelabels are not.
- **The images.** Every image costs (width / 32) x (height / 32) vision tokens. Any other count means something resized it.
- **The length.** No sequence reaches `MAX_SEQ_LENGTH`. The answer sits at the end and would be cut off.

In [14]:
prediction_prompt = tokenizer.apply_chat_template(
    train_examples[0]["messages"][:-1], add_generation_prompt=True, tokenize=False, enable_thinking=False
)
assert prediction_prompt.rstrip("\n").endswith("</think>"), "the prediction prompt leaves the think block open"

partly_masked = next(
    (example for example in train_examples if len(example["trained_dimensions"]) < len(DIMENSIONS)), train_examples[1]
)
most_elongated = max(
    train_examples,
    key=lambda example: max(example["messages"][1]["content"][0]["image"].size)
    / min(example["messages"][1]["content"][0]["image"].size),
)
check_examples = [train_examples[0], partly_masked, most_elongated]
batch, positions = collator.collate(check_examples)
labels, input_ids = batch["labels"], batch["input_ids"]
for row, example in enumerate(check_examples):
    kept = [index for dimension in example["trained_dimensions"] for index in positions[row][dimension]]
    left_out = [
        index
        for dimension in DIMENSIONS
        if dimension not in example["trained_dimensions"]
        for index in positions[row][dimension]
    ]
    assert (labels[row, kept] == input_ids[row, kept]).all(), "a human class value is masked: check instruction_part"
    assert (labels[row, left_out] == -100).all(), "a prelabel is still in the loss"

image_tokens = (input_ids == tokenizer.image_token_id).sum(dim=1).tolist()
expected_tokens = [
    (example["messages"][1]["content"][0]["image"].width // PATCH_SIZE)
    * (example["messages"][1]["content"][0]["image"].height // PATCH_SIZE)
    for example in check_examples
]
assert image_tokens == expected_tokens, f"{image_tokens} vision tokens, expected {expected_tokens}: something resized"

length_examples = random.Random(SEED).sample(train_examples, 64)
lengths = sorted(collator([example])["input_ids"].shape[1] for example in length_examples)
print(f"sequence length: median {lengths[32]}, max {lengths[-1]}, limit {MAX_SEQ_LENGTH}")
assert lengths[-1] < MAX_SEQ_LENGTH, "sequences reach MAX_SEQ_LENGTH: raise it or lower IMAGE_SIZE"

sequence length: median 640, max 640, limit 2048


### Train

AdamW 8-bit with a cosine schedule. The LoRA of the vision tower learns at a quarter of the language rate, since it was pretrained on far more images than we have.

A checkpoint is saved every 500 steps. If the kernel dies, rerun the cells up to here and call `trainer.train(resume_from_checkpoint=True)`.

In [15]:
import bitsandbytes
from trl import SFTConfig, SFTTrainer


def build_optimizer():
    """AdamW 8-bit with the LoRA weights of the vision tower in their own, lower learning rate group."""
    vision, language = [], []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            (vision if ".visual." in name else language).append(parameter)
    assert vision and language, "no trainable weights found under .visual."
    groups = [
        {"params": language, "lr": LANGUAGE_LEARNING_RATE},
        {"params": vision, "lr": VISION_LEARNING_RATE},
    ]
    return bitsandbytes.optim.AdamW8bit(groups, lr=LANGUAGE_LEARNING_RATE, weight_decay=WEIGHT_DECAY)


FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_examples,
    data_collator=collator,
    optimizers=(build_optimizer(), None),
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_ratio=WARMUP_RATIO,
        learning_rate=LANGUAGE_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        logging_steps=25,
        seed=SEED,
        output_dir=str(OUTPUT_DIR),
        report_to="none",
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LENGTH,
    ),
)
trainer_stats = trainer.train()

gpu = torch.cuda.get_device_properties(0)
print(f"{trainer_stats.metrics['train_runtime'] / 60:.0f} minutes of training")
peak_gigabytes = torch.cuda.max_memory_reserved() / 1024**3
print(f"peak memory {peak_gigabytes:.1f} of {gpu.total_memory / 1024**3:.0f} GB on {gpu.name}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,306 | Num Epochs = 3 | Total steps = 2,685
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 101,956,608 of 9,511,770,352 (1.07% trained)


Step,Training Loss
25,0.106236
50,0.053032
75,0.045356
100,0.041884
125,0.040062
150,0.037799
175,0.035931
200,0.037878
225,0.033368
250,0.032602


73 minutes of training
peak memory 38.2 of 79 GB on NVIDIA H100 80GB HBM3


## 7. Save and merge

The LoRA adapters, about 200 MB, go to `ADAPTER_DIR`. vLLM serves the merged 16-bit model, about 18 GB in `MERGED_DIR`. With bf16 LoRA the merge is lossless.

In [16]:
import shutil

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
free_gigabytes = shutil.disk_usage(WORK_DIR).free / 1024**3
assert free_gigabytes > 40, f"{free_gigabytes:.0f} GB free, the merge needs about 40 GB. The adapters are saved."
model.save_pretrained_merged(str(MERGED_DIR), tokenizer)
print(f"adapters in {ADAPTER_DIR}, merged model in {MERGED_DIR}")

Unsloth: Merge process complete. Saved to `/workspace/qwen35_merged`
adapters in /workspace/qwen35_lora, merged model in /workspace/qwen35_merged


## 8. Predict, score, submit

These cells only need the settings cell and section 4. On Saturday after 13:00:

1. Add "final" to `PREDICTED_PARTS` and restart the kernel.
2. Run the settings cell and section 4, then continue here.

The final images are for inference only. Check your speed: you need at least 2 images per second. This model reached about 29 on our H100.

Our outputs below are from a pass over dev and final; the holdout score is from the pass before.

### The final images

The cell asks for the passphrase. Never write it into the notebook: your repository must not contain it.

In [23]:
import getpass

import py7zr

parts = {"holdout": holdout_rows, "dev": dev_rows}
if "final" in PREDICTED_PARTS:
    final_dir = DATA_DIR / "final"
    if not (final_dir / "final.csv").exists():
        passphrase = getpass.getpass("passphrase of the final images: ")
        with py7zr.SevenZipFile(DATA_DIR / "final_encrypted.bin", password=passphrase) as archive:
            archive.extractall(final_dir)
    parts["final"] = read_part(final_dir / "final.csv")
parts = {name: parts[name] for name in PREDICTED_PARTS}
print({name: len(rows) for name, rows in parts.items()})

passphrase of the final images:  ········


{'dev': 5261, 'final': 5292}


### Free the GPU for vLLM

vLLM runs as its own process and needs the GPU memory that this kernel still holds. If the check fails, restart the kernel, run the settings cell and section 4, and continue here.

In [24]:
import gc

import torch

for name in ("trainer", "model", "collator", "train_examples", "batch", "labels", "input_ids"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

assert (MERGED_DIR / "config.json").exists(), "no merged model yet: run section 7"
free_bytes, total_bytes = torch.cuda.mem_get_info()
vllm_memory_fraction = round(min(40 * 1024**3, free_bytes - 8 * 1024**3) / total_bytes, 2)
print(f"{free_bytes / 1024**3:.0f} of {total_bytes / 1024**3:.0f} GB free, vLLM gets {vllm_memory_fraction:.0%}")
assert free_bytes > 36 * 1024**3, "under 36 GB free: restart the kernel as described above"

54 of 79 GB free, vLLM gets 51%


### Predict with vLLM

The cell writes one chat request per image, in the same conversation as in training but without the answer. Then it runs the merged model in the vLLM environment:

- greedy decoding
- the answer held to the JSON schema
- thinking off
- prefix caching, since the system prompt is the same for every image

Loading the model takes a few minutes. The answers land in `PREDICTION_DIR`, one file per part.

In [25]:
import base64
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

VLLM_PREDICTION_SCRIPT = '''
"""Answer chat requests with a local model in vLLM: greedy, held to a JSON schema, thinking off."""

import json
import sys
import time

from vllm import LLM, SamplingParams

try:
    from vllm.sampling_params import StructuredOutputsParams
except ImportError:
    StructuredOutputsParams = None
    from vllm.sampling_params import GuidedDecodingParams


def main():
    model_dir, requests_path, schema_path, predictions_path, memory_fraction = sys.argv[1:]
    schema = json.loads(open(schema_path).read())
    guided = (
        {"structured_outputs": StructuredOutputsParams(json=schema)}
        if StructuredOutputsParams is not None
        else {"guided_decoding": GuidedDecodingParams(json=schema)}
    )
    sampling = SamplingParams(temperature=0.0, max_tokens=120, **guided)
    llm = LLM(
        model=model_dir,
        max_model_len=2048,
        gpu_memory_utilization=float(memory_fraction),
        max_num_batched_tokens=32768,
        max_num_seqs=256,
        enable_prefix_caching=True,
        limit_mm_per_prompt={"image": 1, "video": 0},
    )
    requests = [json.loads(line) for line in open(requests_path)]
    started = time.time()
    with open(predictions_path, "w") as predictions:
        for start in range(0, len(requests), 512):
            chunk = requests[start : start + 512]
            outputs = llm.chat(
                [request["messages"] for request in chunk],
                sampling,
                use_tqdm=False,
                chat_template_kwargs={"enable_thinking": False},
            )
            for request, output in zip(chunk, outputs):
                answer = {"image_id": request["image_id"], "answer": output.outputs[0].text}
                predictions.write(json.dumps(answer) + "\\n")
            done = start + len(chunk)
            print(f"{done} / {len(requests)} images, {done / (time.time() - started):.1f} images/s", flush=True)


if __name__ == "__main__":
    main()
'''


def request_line(row):
    """One chat request for vLLM, or None when the image cannot be read."""
    try:
        image_url = "data:image/jpeg;base64," + base64.b64encode(prepared_jpeg(row["image_file"])).decode()
    except (OSError, ValueError) as error:
        print(f"{row['image_id']} cannot be read, it keeps the constant answer: {error}")
        return None
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": user_prompt(row)},
            ],
        },
    ]
    return json.dumps({"image_id": row["image_id"], "messages": messages})


PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
requests_path = PREDICTION_DIR / "requests.jsonl"
schema_path = PREDICTION_DIR / "response_schema.json"
all_predictions_path = PREDICTION_DIR / "predictions_all.jsonl"
script_path = PREDICTION_DIR / "predict_with_vllm.py"
schema_path.write_text(json.dumps(RESPONSE_SCHEMA))
script_path.write_text(VLLM_PREDICTION_SCRIPT)

with requests_path.open("w") as requests_file, ThreadPoolExecutor(max_workers=16) as pool:
    for name, rows in parts.items():
        for line in tqdm(pool.map(request_line, rows.to_dict("records")), total=len(rows), desc=f"{name} requests"):
            if line is not None:
                requests_file.write(line + "\n")

while not VLLM_READY.exists():
    assert VLLM_LOG.exists(), "the vLLM install never started: run its cell in section 2"
    assert not VLLM_FAILED.exists(), f"the vLLM install failed, end of its log:\n{VLLM_LOG.read_text()[-3000:]}"
    print("waiting for the vLLM install ...", flush=True)
    time.sleep(30)

vllm_environment = {
    name: value
    for name, value in os.environ.items()
    if not name.startswith(("UNSLOTH", "PYTORCH_", "TORCH", "TRITON", "HF_HUB_ENABLE_HF_TRANSFER"))
}
command = [
    str(VLLM_ENV_DIR / "bin" / "python"),
    str(script_path),
    str(MERGED_DIR),
    str(requests_path),
    str(schema_path),
    str(all_predictions_path),
    str(vllm_memory_fraction),
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=vllm_environment)
for output_line in process.stdout:
    print(output_line, end="")
assert process.wait() == 0, "the vLLM prediction failed, see its output above"

part_of_image = {image_id: name for name, rows in parts.items() for image_id in rows["image_id"]}
lines_per_part = {name: [] for name in parts}
for line in all_predictions_path.read_text().splitlines():
    lines_per_part[part_of_image[json.loads(line)["image_id"]]].append(line)
for name, lines in lines_per_part.items():
    (PREDICTION_DIR / f"predictions_{name}.jsonl").write_text("".join(f"{line}\n" for line in lines))
    print(f"{name}: {len(lines)} answers in {PREDICTION_DIR / f'predictions_{name}.jsonl'}")

512 / 10553 images, 13.3 images/s
...
10240 / 10553 images, 28.6 images/s
10553 / 10553 images, 28.7 images/s
dev: 5261 answers in /workspace/predictions/predictions_dev.jsonl
final: 5292 answers in /workspace/predictions/predictions_final.jsonl


### Score the holdout

metric.py is the leaderboard metric: the macro F1 of each dimension, averaged over the six. As on Kaggle, only human labels in relevant cells count. The holdout is other images than dev, so your leaderboard score will differ by a few points.

In [26]:
holdout_answers = read_predictions("holdout")
if holdout_answers:
    solution = pd.DataFrame({"image_id": holdout_rows["image_id"]})
    for dimension in DIMENSIONS:
        scored = (holdout_rows[f"{dimension}_source"] == "human") & (holdout_rows[f"{dimension}_relevant"] == "true")
        solution[dimension] = holdout_rows[dimension].where(scored)
    holdout_submission = pd.DataFrame(
        [
            {"image_id": image_id, **(holdout_answers.get(image_id) or dict.fromkeys(DIMENSIONS))}
            for image_id in holdout_rows["image_id"]
        ]
    )
    metrics = metric.evaluate(solution, holdout_submission, "image_id")
    unparseable = sum(holdout_answers.get(image_id) is None for image_id in holdout_rows["image_id"])
    print(f"holdout, {len(holdout_rows)} images, {unparseable} without a valid answer\n")
    print(f"{'':20} {'F1 macro':>9} {'F1 weighted':>12} {'cells':>6}")
    for dimension in [dimension for dimension in DIMENSIONS if dimension in metrics] + ["overall"]:
        values = metrics[dimension]
        print(f"{dimension:20} {values['f1_macro']:9.3f} {values['f1_weighted']:12.3f} {values['support']:6}")
    weakest = sorted(
        (values["f1"], f"{dimension}/{class_name}", values["support"])
        for dimension in DIMENSIONS
        if dimension in metrics
        for class_name, values in metrics[dimension]["per_class"].items()
    )[:8]
    print("\nweakest classes:", ", ".join(f"{name} {f1:.2f} (n={support})" for f1, name, support in weakest))

holdout, 600 images, 0 without a valid answer

                      F1 macro  F1 weighted  cells
subject                  0.875        0.927    564
background               0.911        0.967    486
perspective              0.859        0.880    358
individuals              0.989        0.998    487
text_classification      0.842        0.970    539
text_language            0.916        0.940    251
overall                  0.899        0.951   2685

weakest classes: text_classification/certification_seal 0.67 (n=2), subject/accessories 0.69 (n=17), perspective/top_down 0.71 (n=30), subject/multiview 0.71 (n=7), text_classification/other 0.75 (n=18), subject/multipack_trade_unit 0.77 (n=8), text_classification/promotional_text 0.78 (n=10), perspective/left_side 0.80 (n=12)


### Write submission.csv

The file has one row per image of sample_submission.csv:

- The dev and final rows take the answers in `PREDICTION_DIR`, from this run or an earlier one.
- A row without an answer keeps the constant answer of sample_submission.csv. So before Saturday 13:00, your final rows are the constants.

Kaggle rejects empty cells and unknown class names, so the cell checks both. Submit the file with the command it prints.

In [27]:
answers = {**read_predictions("dev"), **read_predictions("final")}
answers = {image_id: answer for image_id, answer in answers.items() if answer is not None}
submission = sample_submission.set_index("image_id")
predicted_ids = [image_id for image_id in submission.index if image_id in answers]
if predicted_ids:
    submission.loc[predicted_ids, list(DIMENSIONS)] = [
        [answers[image_id][dimension] for dimension in DIMENSIONS] for image_id in predicted_ids
    ]
submission = submission.reset_index()

assert submission["image_id"].tolist() == sample_submission["image_id"].tolist()
for dimension in DIMENSIONS:
    assert submission[dimension].isin(CLASSES[dimension]).all(), f"{dimension} has an empty cell or an unknown class"

dev_ids = set(dev_rows["image_id"])
predicted_dev = sum(image_id in dev_ids for image_id in predicted_ids)
predicted_final = len(predicted_ids) - predicted_dev
final_count = len(submission) - len(dev_ids)
submission_path = PREDICTION_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"{submission_path}: dev {predicted_dev} of {len(dev_ids)} predicted, final {predicted_final} of {final_count}")
print(f'kaggle competitions submit -c {COMPETITION} -f {submission_path} -m "Qwen 3.5 9B reference"')

/workspace/predictions/submission.csv: dev 5261 of 5261 predicted, final 5292 of 5292
kaggle competitions submit -c hackapertus-galaxus -f /workspace/predictions/submission.csv -m "Qwen 3.5 9B reference"


### Publish the weights (optional)

The hand-in asks for the weights behind your final submission, for example on Hugging Face. This cell uploads the adapters to a public repository. Note the printed revision for section 7 of your technical report.

In [28]:
from huggingface_hub import HfApi, get_token, login

HF_REPO_ID = None  # for example "your-team/galaxus-image-types-lora"

if HF_REPO_ID:
    if get_token() is None:
        login()
    hub = HfApi()
    hub.create_repo(HF_REPO_ID, exist_ok=True)
    commit = hub.upload_folder(repo_id=HF_REPO_ID, folder_path=str(ADAPTER_DIR), commit_message="Qwen 3.5 9B LoRA")
    print(f"{commit.commit_url}\nrevision {commit.oid}")